- ### Threading and processing

1. **Threading**

**[Multi-threading -- for I/O intensive task (e.g. reading/writing files)]**

multi-threading

thread pool executor

multi-threading queue

2. **Processing**

**[Multi-processing -- for CPU intensive task (e.g. calculation)]**

multi-processing

process pool executor

multi-processing queue

3. **Asyncio**

 **[Asyncio (single threading) -- for I/O intensive task (e.g. network service)]**

 coroutine

 event loop

 task

 future

---

- ### Threading

**start & join thread**

In [29]:
import threading
import time

def print_numbers():
    for i in range(6):
        print(i, end=' ')
        time.sleep(1)

thread = threading.Thread(target=print_numbers)
thread.start()
thread.join()
print("Finished")

0 1 2 3 4 5 Finished


**multi-threading**

In [ ]:
import threading
import time

def worker(num):
    print(f"thread {num} starting")
    time.sleep(num+1)
    print(f"thread {num} ending")

threads = []
for i in range(3):
    t = threading.Thread(target=worker, args=(2*i,))
    threads.append(t)
    t.start()
    time.sleep(i)
print(thread)
for t in threads:
    t.join()

thread 0 starting
thread 2 starting
thread 0 ending
thread 4 starting
thread 2 ending
<Thread(Thread-12 (print_numbers), stopped 20188)>
thread 4 ending


**thread pool executor**

In [8]:
from concurrent.futures import ThreadPoolExecutor
import time

def task(n):
    print(f"Task {n} starting")
    time.sleep(n+1)
    print(f"Task {n} completed")
    return n*n

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = []
    for i in range(3):
        f = executor.submit(task, i)
        time.sleep(2*i+1)
        futures.append(f)
    
    for f in futures:
        print(f.result(), end=' ')

Task 0 starting
Task 0 completed
Task 1 starting
Task 1 completed
Task 2 starting
Task 2 completed
0 1 4 

**multi-threading queue**

In [1]:
import queue
import threading
import time

# create a FIFO queue with optional max size
q = queue.Queue(maxsize=1)

def producer():
    for i in range(5):
        print("Producing", i)
        q.put(i)               # will block if full
        time.sleep(1)

def consumer():
    while True:
        time.sleep(2)
        item = q.get()         # blocks if empty
        print("Consuming", item)
        q.task_done()          # mark as processed

# start consumer thread
threading.Thread(target=consumer).start()

# run producer in main thread
producer()

# wait until all items processed
q.join()
print("All done")


Producing 0
Producing 1
Consuming 0
Producing 2
Consuming 1
Producing 3
Consuming 2
Producing 4
Consuming 3
Consuming 4
All done


---

- ### Processing

**multiprocessing**

In [ ]:
# silent when running in Jupyter, use in script
import multiprocessing
import time

def worker(num):
    print(f"Process {num} starting")
    time.sleep(num+1)
    print(f"Process {num} ending")

if __name__ == '__main__':
    processes = []
    for i in range(5):
        p = multiprocessing.Process(target=worker, args=(i,))
        processes.append(p)
        p.start()
        print(i, end=' ')
        time.sleep(i)
    print(processes)
    for p in processes:
        p.join()

0 1 2 3 4 [<Process name='Process-44' pid=22032 parent=12336 stopped exitcode=1>, <Process name='Process-45' pid=5448 parent=12336 stopped exitcode=1>, <Process name='Process-46' pid=10588 parent=12336 stopped exitcode=1>, <Process name='Process-47' pid=16396 parent=12336 stopped exitcode=1>, <Process name='Process-48' pid=28356 parent=12336 stopped exitcode=1>]


**process pool executor**

In [ ]:
# silent when running in Jupyter, use in script
from concurrent.futures import ProcessPoolExecutor
import time

def task(n):
    print(f"Task {n} starting")
    time.sleep(n+1)
    print(f"Task {n} completed")
    return n*n
if __name__ == '__main__':
    with ProcessPoolExecutor(max_workers=5) as executor:
        futures = []
        for i in range(5):
            f = executor.submit(task, i)
            futures.append(f)

        for f in futures:
            print(f.result())

**multi-processing queue**

In [ ]:
# silent when running in Jupyter, use in script
from multiprocessing import Process, Queue
import time

def worker(q):
    for i in range(5):
        time.sleep(1)
        q.put(i)
        print(f"Task {i} in process")

if __name__ == "__main__":
    q = Queue()
    p = Process(target=worker, args=(q,))
    p.start()
    for i in range(5):
        item = q.get()
    p.join()


---

- ### Asyncio

**coroutine & event loop**

In [43]:
import asyncio
import nest_asyncio # only needed in Jupyter
import sys, io, contextlib

nest_asyncio.apply() # only needed in Jupyter

# define an async task
async def task(n):
    print(f"Task {n} starting")
    await asyncio.sleep(n+1)
    print(f"Task {n} completed")
    return n

# run multiple tasks concurrently
async def main():
    await asyncio.gather(*(task(i) for i in range(5)))
    f = io.StringIO()
    with contextlib.redirect_stdout(f):
        global results
        results = await asyncio.gather(*(task(i) for i in range(5)))
    
# run the main function
asyncio.run(main())
print(results)

Task 0 starting
Task 1 starting
Task 2 starting
Task 3 starting
Task 4 starting
Task 0 completed
Task 1 completed
Task 2 completed
Task 3 completed
Task 4 completed
[0, 1, 2, 3, 4]


**task**

In [22]:
import asyncio
import nest_asyncio # only needed in Jupyter

nest_asyncio.apply() # only needed in Jupyter

async def zero(n):
    print(next(iterator0), end=' ')

async def odd(n):
    print(next(iterator1), end=' ')

async def even(n):
    print(next(iterator2), end=' ')

async def main(n):
    global iterator0, iterator1, iterator2
    iterator0 = iter(n*[0])
    iterator1 = iter(range(1, n+1, 2))
    iterator2 = iter(range(2, n+1, 2))
    task = []
    for _ in range(n//2):
         task.append(asyncio.create_task(zero(n)))
         await asyncio.sleep(0.1)

         task.append(asyncio.create_task(odd(n)))
         await asyncio.sleep(0.1)

         task.append(asyncio.create_task(zero(n)))
         await asyncio.sleep(0.1)

         task.append(asyncio.create_task(even(n)))
         await asyncio.sleep(0.1)

    if n % 2 == 1:
        task.append(asyncio.create_task(zero(n)))
        await asyncio.sleep(0.1)

        task.append(asyncio.create_task(odd(n)))
        await asyncio.sleep(0.1)

asyncio.run(main(10))

0 1 0 2 0 3 0 4 0 5 0 6 0 7 0 8 0 9 0 10 

**future**

In [46]:
import asyncio

async def main():
    future = asyncio.Future()
    future.set_result("Hello from a future")
    print(await future)

asyncio.run(main())

Hello from a future
